# 04 · TabICL v2 (Colab GPU)

TabICL v2 is a pretrained in-context learner for tables (BSD-3 license). Nothing is trained here: the model reads labeled context rows and predicts query rows in one forward pass.

**What this notebook does.** It tunes the inference settings of the model on the training season, predicts the held-out evaluation panel, and saves every prediction to Google Drive. It computes no held-out score: the panel files it reads have no outcome column, and scoring happens later on the local machine in `05_model_comparison.ipynb`.

**What it needs.** One file on Drive: `MyDrive/shotlab/shotlab_bundle.zip`, built locally with `python -m shotlab.bundle`. The bundle holds the `shotlab` package, the frozen configs, the feature tables and their checksums.

**Restart-safe.** Predictions are written in 2,000-row chunks under a hash of the full run configuration. Re-running any cell skips finished chunks, so a disconnected session loses at most one chunk.

## 0. Setup

In [ ]:
MODEL = "tabicl"
DRIVE = "/content/drive/MyDrive/shotlab"

import os
import pathlib
import subprocess
import sys
import zipfile

from google.colab import drive

drive.mount("/content/drive")
ROOT = pathlib.Path("/content/shotlab")
with zipfile.ZipFile(f"{DRIVE}/shotlab_bundle.zip") as archive:
    archive.extractall(ROOT)
os.environ.update(SHOTLAB_ROOT=str(ROOT), SHOTLAB_ARTIFACTS=f"{DRIVE}/artifacts", SHOTLAB_BUNDLE=str(ROOT / "artifacts" / "bundle"))
sys.path.insert(0, str(ROOT / "src"))
sorted(p.name for p in (ROOT / "artifacts" / "bundle").iterdir())

In [ ]:
# Same versions as the local machine. torch is left as Colab ships it (the CUDA build); its version is recorded with every run.
PINS = ["numpy==2.5.5", "pandas==3.1.1", "pyarrow==26.0.1", "scikit-learn==1.9.1", "pyyaml==6.0.3", "tabicl==2.2.0"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)

import importlib.metadata
import numpy
import pandas

if (numpy.__version__, pandas.__version__) != (importlib.metadata.version("numpy"), importlib.metadata.version("pandas")):
    print("New numpy/pandas installed. The runtime restarts now; run the notebook again from the top.")
    os.kill(os.getpid(), 9)

In [ ]:
import json
import time

import numpy as np
import pandas as pd
import torch
import yaml

from shotlab import bundle, inference, store
from shotlab.data import ARTIFACTS, CONFIGS, config, load_grid

pd.set_option("display.width", 220, "display.max_columns", 30)
info = bundle.verify(inference.BUNDLE)
for season in ("S1", "S2", "T"):
    assert "outcome" not in pd.read_parquet(inference.BUNDLE / f"panel_{season}.parquet").columns
assert torch.cuda.is_available(), "Switch the runtime to a GPU: Runtime > Change runtime type."
gpu = torch.cuda.get_device_properties(0)
print("bundle built from commit", info["git_commit"][:7], "· checksums verified · query files carry no outcome")
print("GPU:", gpu.name, f"{gpu.total_memory / 2**30:.0f} GB")
store.environment()

## 1. Timing pilot

Training-season rows only: month block 0 supplies 2,000 queries, the other blocks supply the context. The pilot answers two questions before anything expensive runs: how large a context fits in GPU memory, and how long the frozen grid would take.

In [ ]:
PILOT_SIZES = [4000, 16000, 64000, 128000, 10**9]
pilot_path = ARTIFACTS / "pilot" / f"{MODEL}.csv"
if pilot_path.exists():
    pilot = pd.read_csv(pilot_path)
else:
    runs = []
    for variant in [{"kv_cache": False}, {"kv_cache": True}]:
        runs.append(inference.pilot(MODEL, "S1_to_S2", PILOT_SIZES, n_estimators=8, queries=2000, options=variant).assign(variant=json.dumps(variant)))
    pilot = pd.concat(runs, ignore_index=True)
    pilot_path.parent.mkdir(parents=True, exist_ok=True)
    pilot.to_csv(pilot_path, index=False)
pilot.round(3)

## 2. Run settings

The grid was frozen before the pilot. `docs/freeze.md` allows one kind of change here: cutting the grid or capping the context if the pilot shows the frozen grid cannot run. Any change is written into the config below, mirrored in the repository config and logged in the freeze change log, all before a held-out score exists.

In [ ]:
OVERRIDES = {}

spec = load_grid(MODEL)
for section, values in OVERRIDES.items():
    spec[section].update(values)
(CONFIGS / f"{MODEL}.yaml").write_text(yaml.safe_dump(spec, sort_keys=False))
print(yaml.safe_dump(load_grid(MODEL), sort_keys=False))

In [ ]:
def projected_hours(pilot, spec, queries=config()["tuning_query_size"], folds=5):
    """Rough cost of the tuning grid: one fit plus chunked prediction per configuration and fold."""
    options = spec.get("options", {})
    matches = pilot["variant"].map(lambda text: all(options.get(k) == v for k, v in json.loads(text).items()))
    ok = pilot[matches & (pilot["status"] == "ok")].sort_values("context")
    seconds = 0.0
    for point in inference.grid_points(spec, spec.get("options", {}).get("max_context")):
        rows = min(point["context_rows"], ok["context"].max())
        fit = np.interp(rows, ok["context"], ok["fit_s"])
        predict = np.interp(rows, ok["context"], ok["predict_s"]) * queries / 2000
        seconds += folds * (fit + predict) * point["n_estimators"] / 8
    return seconds / 3600


spec = load_grid(MODEL)
points = inference.grid_points(spec, spec.get("options", {}).get("max_context"))
print(f"{len(points)} configurations x 5 folds; projected {projected_hours(pilot, spec):.1f} GPU-hours per direction for tuning")

## 3. Primary run

For each direction (train S1 → predict S2, and the reverse):

1. **Tune** on the training season with the same five month blocks the trained models use. Context rows come from four blocks, queries from the fifth. Smoothed shooter and defender rates are cross-fitted, so no context row carries its own outcome.
2. **Select** the simplest configuration within one standard error of the best log loss.
3. **Calibrate** with a two-parameter Platt step fitted on the out-of-fold predictions of the selected configuration.
4. **Predict** the 50,000-shot evaluation panel of the other season, once per context seed.

In [ ]:
log_path = ARTIFACTS / "logs" / f"{MODEL}.log"
log_path.parent.mkdir(parents=True, exist_ok=True)


def progress(line):
    stamped = f"{time.strftime('%m-%d %H:%M:%S')} {line}"
    print(stamped)
    with open(log_path, "a") as handle:
        handle.write(stamped + "\n")


inference.run_suite("primary", MODEL, progress=progress)

### What tuning chose

Scores below are month-blocked scores inside the training season. They say nothing yet about the held-out season.

In [ ]:
rows, tables = [], {}
for direction in inference.CROSS_SEASON:
    selection = inference.load_selection(MODEL, direction)
    rows.append({"direction": direction, **selection["best_params"], "cv_log_loss": selection["cv_log_loss_selected"],
                 "cv_log_loss_best": selection["cv_log_loss_best"], "cv_auc": selection["cv_auc_selected"],
                 "platt_slope": selection["platt"]["slope"], "platt_intercept": selection["platt"]["intercept"],
                 "configurations": selection["configurations"], "tuning_hours": selection["tuning_seconds"] / 3600})
    tables[direction] = pd.read_parquet(inference.selection_path(inference.primary_config(MODEL, direction)).with_suffix(".parquet"))
pd.DataFrame(rows).set_index("direction").T

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, (direction, table) in zip(axes, tables.items()):
    mean = table.groupby(["param_context_rows", "param_n_estimators", "param_sampling", "param_m"])["log_loss"].mean()
    best_m = mean.groupby(level=[0, 1, 2]).min().reset_index()
    sizes = sorted(best_m["param_context_rows"].unique())
    for (members, sampling), part in best_m.groupby(["param_n_estimators", "param_sampling"]):
        ax.plot([sizes.index(size) for size in part["param_context_rows"]], part["log_loss"], marker="o",
                label=f"{members} members, {sampling}")
    ax.set_xticks(range(len(sizes)), ["all" if size >= 10**8 else f"{size // 1000}k" for size in sizes])
    ax.set_title(f"{direction}: inner CV log loss (best m)")
    ax.set_xlabel("context rows")
axes[0].set_ylabel("log loss")
axes[0].legend(frameon=False, fontsize=8)
plt.tight_layout()

## 4. Explanation probes

Predictions on 2,000 probe shots with feature blocks shuffled and single features shifted. The same probe and the same perturbations are applied to all four models; importance and effect curves are computed locally, where the outcomes are.

In [ ]:
inference.run_suite("explain", MODEL, progress=progress)

## 5. Side arms

Structure settings stay at the primary selection. Only the smoothing strength `m` is re-tuned, and only where the arm contains smoothed rates.

- **Raw IDs:** shooter and defender IDs as plain categorical columns instead of smoothed rates.
- **Ablation ladder:** location → + shot type → + creation → + defense → + closeout, and the primary set plus team and month.
- **Learning curve:** nested training subsets of 500 to 100,000 shots, the whole subset as context.

In [ ]:
SIDE_ARMS = ["raw_ids", "ablation", "learning_curve"]
for suite in SIDE_ARMS:
    inference.run_suite(suite, MODEL, progress=progress)

## 6. Test season (optional)

Context: both labeled seasons. Queries: the whole unlabeled test season. Run only if the pilot shows it is affordable; otherwise the trained models supply the test predictions.

In [ ]:
RUN_TEST_SEASON = False
if RUN_TEST_SEASON:
    inference.run_suite("final", MODEL, progress=progress)

## 7. Pack the results

Parquet, JSON and CSV only. No model objects and no pickles cross machines.

In [ ]:
import hashlib

catalog = store.catalog()
catalog = catalog[catalog["model"] == MODEL]
display(catalog.groupby(["arm", "direction", "role"])["rows"].agg(["size", "sum"]).rename(columns={"size": "files", "sum": "rows"}))
out = pathlib.Path(DRIVE) / f"results_{MODEL}.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for folder in ("predictions", "tuning", "explain"):
        for path in sorted((ARTIFACTS / folder / MODEL).glob("*")):
            archive.write(path, f"{folder}/{MODEL}/{path.name}")
    for path in (pilot_path, log_path, CONFIGS / f"{MODEL}.yaml"):
        archive.write(path, f"colab/{MODEL}/{path.name}")
print(out, f"{out.stat().st_size / 2**20:.1f} MB", "sha256", hashlib.sha256(out.read_bytes()).hexdigest())